<font size=10>**NETWORK CHARACTERIZATION**</font>

**Following Sturm et al. (2025), Sections 4 & 6**

<font color='#BFD72F' size=5>**FOCUS**:</font> Degree distributions, hierarchical structure, community detection

*"The observed exponential distribution suggests networks that lack hubs typically found in scale-free networks."*

*"A negative correlation between clustering coefficient and degree suggests hierarchical organization."*

<font color='#BFD72F' size=6>**TABLE OF CONTENTS**</font> <a class="anchor" id='toc'></a>
- [1. Imports & Load Network](#1-imports)
- [2. Degree Distribution Analysis](#2-degree-dist)
- [3. Distribution Fitting (Exponential vs Power-Law)](#3-fitting)
- [4. Hierarchical Structure: C(k) vs k](#4-hierarchical)
- [5. Community Detection (Louvain)](#5-communities)
- [6. Network Visualization](#6-visualization)
- [7. Assortativity & Correlations](#7-assortativity)

# <font color='#BFD72F' size=6>**1. Imports & Load Network**</font> <a class="anchor" id="1-imports"></a>
[Back to TOC](#toc)

In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.optimize import curve_fit
import warnings
import os

try:
    import powerlaw
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'powerlaw'])
    import powerlaw

try:
    import community as community_louvain
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'python-louvain'])
    import community as community_louvain

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
print('Libraries loaded.')

In [ ]:
# Load network
G = nx.read_graphml('../graphs/cobidding_network.graphml')
print(f'Network loaded: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')
print(f'Density: {nx.density(G):.6f}')
print(f'Is connected: {nx.is_connected(G)}')

# If not connected, use giant component
if not nx.is_connected(G):
    components = sorted(nx.connected_components(G), key=len, reverse=True)
    G = G.subgraph(components[0]).copy()
    print(f'Using Giant Component: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')

# <font color='#BFD72F' size=6>**2. Degree Distribution Analysis**</font> <a class="anchor" id="2-degree-dist"></a>
[Back to TOC](#toc)

Paper Figure 2-B: *"Colored dots show the degree distribution of the inferred networks and the best-fitted exponential distribution."*

In [ ]:
# Compute degree sequence
degrees = np.array([d for _, d in G.degree()])

print(f'Degree statistics:')
print(f'  Min: {degrees.min()}')
print(f'  Max: {degrees.max()}')
print(f'  Mean: {degrees.mean():.2f}')
print(f'  Median: {np.median(degrees):.0f}')
print(f'  Std: {degrees.std():.2f}')

# Complementary CDF (CCDF)
sorted_degrees = np.sort(degrees)
ccdf = 1 - np.arange(1, len(sorted_degrees) + 1) / len(sorted_degrees)

# Plot CCDF on log-log scale
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
ax.scatter(sorted_degrees, ccdf, s=15, alpha=0.6, label='Empirical')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Degree k', fontsize=12)
ax.set_ylabel('P(K ≥ k)', fontsize=12)
ax.set_title('Degree Distribution (CCDF)', fontsize=14)
ax.legend()
plt.tight_layout()
os.makedirs('../figures', exist_ok=True)
plt.savefig('../figures/degree_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: ../figures/degree_distribution.png')

# <font color='#BFD72F' size=6>**3. Distribution Fitting**</font> <a class="anchor" id="3-fitting"></a>
[Back to TOC](#toc)

Paper: *"The best-fitted exponential distribution curve. This contrasts with the power-law, which according to Vuong's Closeness Test, shows a less accurate fit."*

We fit both exponential and power-law and compare using likelihood ratio.

In [ ]:
# Fit using powerlaw library
fit = powerlaw.Fit(degrees, discrete=True, verbose=False)

print('=== POWER-LAW FIT ===')
print(f'  Alpha (exponent): {fit.alpha:.3f}')
print(f'  x_min: {fit.xmin}')
print(f'  KS statistic: {fit.power_law.KS():.4f}')

print('\n=== COMPARISON: Power-law vs Exponential ===')
R, p = fit.distribution_compare('power_law', 'exponential', normalized_ratio=True)
print(f'  Likelihood ratio R: {R:.4f}')
print(f'  p-value: {p:.4f}')
if R > 0:
    print('  → Power-law is preferred')
else:
    print('  → EXPONENTIAL is preferred (matches paper finding)')

print('\n=== COMPARISON: Power-law vs Lognormal ===')
R2, p2 = fit.distribution_compare('power_law', 'lognormal', normalized_ratio=True)
print(f'  Likelihood ratio R: {R2:.4f}')
print(f'  p-value: {p2:.4f}')

In [ ]:
# Plot with fitted distributions
fig, ax = plt.subplots(figsize=(8, 6))

# Empirical CCDF
ax.scatter(sorted_degrees, ccdf, s=15, alpha=0.5, color='steelblue', label='Empirical')

# Exponential fit
lam = 1 / degrees.mean()  # MLE for exponential rate
k_range = np.linspace(1, degrees.max(), 200)
exp_ccdf = np.exp(-lam * k_range)
ax.plot(k_range, exp_ccdf, 'r--', linewidth=2, label=f'Exponential (λ={lam:.4f})')

# Power-law fit
alpha = fit.alpha
xmin = fit.xmin
pl_ccdf = (k_range / xmin) ** (1 - alpha)
pl_ccdf = pl_ccdf * ccdf[np.searchsorted(sorted_degrees, xmin)]  # normalize
ax.plot(k_range[k_range >= xmin], pl_ccdf[k_range >= xmin], 'g:', linewidth=2, 
        label=f'Power-law (α={alpha:.2f})')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Degree k', fontsize=12)
ax.set_ylabel('P(K ≥ k)', fontsize=12)
ax.set_title('Degree Distribution: Exponential vs Power-Law Fit', fontsize=14)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('../figures/degree_dist_fits.png', dpi=150, bbox_inches='tight')
plt.show()

# <font color='#BFD72F' size=6>**4. Hierarchical Structure: C(k) vs k**</font> <a class="anchor" id="4-hierarchical"></a>
[Back to TOC](#toc)

Paper Figure 2-C: *"The negative trend between average clustering coefficient of nodes as a function of their degree k along with the best power-law fits."*

A relationship $C(k) \\sim k^{-\\beta}$ indicates hierarchical organization (Ravasz et al., 2002).

In [ ]:
# Compute clustering coefficient for each node
clustering_dict = nx.clustering(G)

# Create DataFrame of (degree, clustering)
node_stats = pd.DataFrame({
    'degree': [G.degree(n) for n in G.nodes()],
    'clustering': [clustering_dict[n] for n in G.nodes()]
})

# Bin by degree and compute average C(k)
node_stats['degree_bin'] = pd.cut(node_stats['degree'], bins=20)
binned = node_stats.groupby('degree_bin').agg(
    avg_degree=('degree', 'mean'),
    avg_clustering=('clustering', 'mean'),
    count=('degree', 'count')
).dropna().reset_index(drop=True)

# Filter bins with enough data
binned = binned[binned['count'] >= 5]

# Log-log fit: log(C) = -β * log(k) + const
log_k = np.log(binned['avg_degree'].values)
log_c = np.log(binned['avg_clustering'].values + 1e-10)

# OLS fit
slope, intercept, r_value, p_value, std_err = stats.linregress(log_k, log_c)
beta = -slope

print(f'Hierarchical structure test: C(k) ~ k^(-β)')
print(f'  β = {beta:.3f}')
print(f'  R² = {r_value**2:.3f}')
print(f'  p-value = {p_value:.2e}')
if beta > 0 and p_value < 0.05:
    print('  → HIERARCHICAL structure confirmed (negative C-k relationship)')
else:
    print('  → No clear hierarchical structure detected')

In [ ]:
# Plot C(k) vs k
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(binned['avg_degree'], binned['avg_clustering'], s=60, color='darkorange', 
           alpha=0.7, edgecolors='black', linewidth=0.5, label='Binned data')

# Fitted line
k_fit = np.linspace(binned['avg_degree'].min(), binned['avg_degree'].max(), 100)
c_fit = np.exp(intercept) * k_fit ** slope
ax.plot(k_fit, c_fit, 'k--', linewidth=2, label=f'Fit: C(k) ~ k$^{{-{beta:.2f}}}$ (R²={r_value**2:.2f})')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Degree k', fontsize=12)
ax.set_ylabel('Average Clustering C(k)', fontsize=12)
ax.set_title('Hierarchical Structure: C(k) vs k', fontsize=14)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('../figures/hierarchical_structure.png', dpi=150, bbox_inches='tight')
plt.show()

# <font color='#BFD72F' size=6>**5. Community Detection (Louvain)**</font> <a class="anchor" id="5-communities"></a>
[Back to TOC](#toc)

Paper: *"Colors represent the communities identified using the Louvain algorithm. Modularity of 0.71-0.80."*

In [ ]:
# Run Louvain community detection
partition = community_louvain.best_partition(G, weight='weight', random_state=42)

# Modularity
modularity = community_louvain.modularity(partition, G, weight='weight')

# Community sizes
community_sizes = pd.Series(partition).value_counts().sort_values(ascending=False)

print(f'Community Detection Results:')
print(f'  Number of communities: {community_sizes.nunique()}')
print(f'  Modularity: {modularity:.4f}')
print(f'  Median community size: {community_sizes.median():.0f}')
print(f'  Q1 community size: {community_sizes.quantile(0.25):.0f}')
print(f'  Q3 community size: {community_sizes.quantile(0.75):.0f}')
print(f'\nTop 10 communities by size:')
print(community_sizes.head(10))

In [ ]:
# Compare with Erdős-Rényi random reference
print('\nComparing modularity with ER random reference...')
n_random = 10
random_modularities = []

for i in range(n_random):
    G_random = nx.gnm_random_graph(G.number_of_nodes(), G.number_of_edges(), seed=i)
    if G_random.number_of_edges() > 0:
        part_random = community_louvain.best_partition(G_random, random_state=i)
        mod_random = community_louvain.modularity(part_random, G_random)
        random_modularities.append(mod_random)

mean_random_mod = np.mean(random_modularities)
print(f'  Real network modularity: {modularity:.4f}')
print(f'  Mean ER random modularity: {mean_random_mod:.4f} ± {np.std(random_modularities):.4f}')
print(f'  Ratio (real/random): {modularity/mean_random_mod:.2f}x')
print(f'  → Network has {"SIGNIFICANTLY higher" if modularity > 2*mean_random_mod else "higher"} modularity than random')

# <font color='#BFD72F' size=6>**6. Network Visualization**</font> <a class="anchor" id="6-visualization"></a>
[Back to TOC](#toc)

Paper Figure 2-A: *"Graphical representation of the giant component. Colors represent communities."*

In [ ]:
# Visualization
# Color nodes by community (top 10 get colors, rest gray)
top_communities = community_sizes.head(10).index.tolist()
colors = plt.cm.tab10(np.linspace(0, 1, 10))
color_map = {comm: colors[i] for i, comm in enumerate(top_communities)}

node_colors = [color_map.get(partition[n], [0.7, 0.7, 0.7, 1.0]) for n in G.nodes()]
node_sizes = [np.log(G.degree(n) + 1) * 10 for n in G.nodes()]

# Layout (spring for smaller graphs, random for very large)
if G.number_of_nodes() < 2000:
    pos = nx.spring_layout(G, k=1/np.sqrt(G.number_of_nodes()), iterations=50, seed=42)
else:
    pos = nx.random_layout(G, seed=42)

fig, ax = plt.subplots(figsize=(14, 10))
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=node_sizes, alpha=0.7, ax=ax)
nx.draw_networkx_edges(G, pos, alpha=0.05, width=0.3, ax=ax)
ax.set_title('Co-Bidding Network (colored by Louvain community)', fontsize=14)
ax.axis('off')
plt.tight_layout()
plt.savefig('../figures/network_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

# <font color='#BFD72F' size=6>**7. Assortativity & Correlations**</font> <a class="anchor" id="7-assortativity"></a>
[Back to TOC](#toc)

In [ ]:
# Degree assortativity
assort = nx.degree_assortativity_coefficient(G)
print(f'Degree assortativity coefficient: {assort:.4f}')
if assort > 0:
    print('  → Assortative: high-degree nodes tend to connect to high-degree nodes')
else:
    print('  → Disassortative: high-degree nodes tend to connect to low-degree nodes')

# Average neighbor degree
avg_neighbor_deg = nx.average_neighbor_degree(G)
knn = pd.DataFrame({'degree': [G.degree(n) for n in G.nodes()],
                    'avg_neighbor_degree': [avg_neighbor_deg[n] for n in G.nodes()]})

# Plot k_nn(k)
binned_knn = knn.groupby(pd.cut(knn['degree'], bins=15)).agg(
    avg_k=('degree', 'mean'), avg_knn=('avg_neighbor_degree', 'mean')
).dropna()

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(binned_knn['avg_k'], binned_knn['avg_knn'], s=60, color='purple', alpha=0.7)
ax.set_xlabel('Degree k', fontsize=12)
ax.set_ylabel('Average Neighbor Degree k_nn(k)', fontsize=12)
ax.set_title(f'Degree-Degree Correlations (r={assort:.3f})', fontsize=14)
plt.tight_layout()
plt.savefig('../figures/assortativity.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Save community assignments
comm_df = pd.DataFrame([
    {'firm_nif': node, 'community': partition[node]} for node in G.nodes()
])
comm_df.to_csv('../data/community_assignments.csv', index=False)
print(f'Saved community assignments: ../data/community_assignments.csv ({len(comm_df)} nodes)')